# CSE476 — Agentic AI and Intelligent Automation
## Project 1 — Study Planner Agent

This Jupyter Notebook contains the **complete, self-contained implementation** of the Study Planner Agent. The instructor can run this notebook from top to bottom without relying on external Python module imports.

### Core Concepts Demonstrated:
1. **One AI Agent**: An intelligent study planning assistant using Gemini (`gemini-2.5-flash`).
2. **Two Tools**: Python functions `add_task` (saves a task and deadline to memory) and `build_schedule` (creates the chronological timeline).
3. **Plan-Act Loop**: A custom `while True` loop showing the step-by-step trace of how the agent decides what to do, calls the tools, receives the output, and updates its plan.
4. **Session Memory**: Dialogue history tracking and task list memory stored across multiple turns in the same conversation.

---  
## Cell 2 — Project Overview

### 1. What the Agent Does
The Study Planner Agent helps students organize their study time. When given exams and assignment deadlines, it registers them, sorts them by due date, and designs a day-by-day study calendar leading up to each due date.

### 2. The Two Tools
- `add_task(name, due)`: Takes the name of a task and its deadline, then appends it to the task list.
- `build_schedule()`: Accesses all stored tasks, parses their deadlines, sorts them, and allocates sequential study blocks beginning from a baseline date (August 27, 2026).

### 3. The Memory System
- **Conversational Memory**: A list of `Content` messages in the agent that maintains the chat history. The entire history is passed to Gemini on each call so it remembers the context of previous messages.
- **Task State Memory**: A simple Python list `tasks` that stores the structured task dicts added during the session.

### 4. Agent vs. Chatbot
A standard chatbot simply receives a prompt and prints the direct response from the model. In contrast, this agent executes a **Plan-Act loop**: it dynamically decides which tools are required, halts to execute them in Python, receives the outputs, updates its memory, and calls the model again. This ensures that the agent uses actual code executions to formulate its plans, rather than just guessing dates or schedules.

---  
## Cell 3 — Imports and Configuration

We load all required libraries and initialize dotenv to configure the API key securely. Make sure your `.env` file containing `GEMINI_API_KEY` is present in the project root folder.

In [ ]:
import os
from datetime import datetime, timedelta
from google import genai
from google.genai import types
from dotenv import load_dotenv

# Load environment variables from the .env file
load_dotenv()
print("Imports and configuration loaded successfully!")

---  
## Cell 4 — Memory

We define the session task memory. This stores the list of dictionary objects representing our tasks. We also provide utility functions to add, retrieve, and clear these tasks.

In [ ]:
# In-memory storage for tasks and deadlines during the session
tasks = []

def add_task_to_memory(name: str, due: str) -> str:
    """
    Add a task and its deadline to the in-memory storage.
    """
    tasks.append({
        "name": name.strip(),
        "due": due.strip()
    })
    return f"Success: Task '{name}' with due date '{due}' has been saved to memory."

def get_tasks_from_memory() -> list:
    """
    Retrieve all tasks currently stored in memory.
    """
    return tasks

def clear_tasks_in_memory() -> str:
    """
    Clear all tasks from memory.
    """
    global tasks
    tasks = []
    return "Success: Task memory has been cleared."

print("Memory storage functions initialized!")

---  
## Cell 5 — Tool 1: add_task

This tool adds a task and its deadline to the memory storage. Gemini will invoke this tool when it recognizes a study deadline mentioned by the user.

In [ ]:
def add_task(name: str, due: str) -> str:
    """
    Adds a study task and its due date/deadline to the agent's memory.
    
    Args:
        name: The name of the task (e.g., 'Java Exam', 'DBMS Assignment').
        due: The due date/deadline of the task (e.g., 'September 5', '2026-09-02').
    """
    return add_task_to_memory(name, due)

---  
## Cell 6 — Tool 2: build_schedule

This tool retrieves all saved tasks, sorts them chronologically using a date helper, and schedules study blocks. It anchors the timeline start to August 27, 2026, to ensure that the schedule generates correctly and reproducibly regardless of when the code is executed.

In [ ]:
def parse_due_date(due_str: str) -> datetime:
    """
    Parses multiple date formats (e.g. 'September 5', '2026-09-02') to a datetime object.
    Defaults to August 2026 environment contexts.
    """
    due_clean = due_str.strip().lower()
    current_year = datetime.now().year
    
    for fmt in ("%Y-%m-%d", "%d-%m-%Y", "%m/%d/%Y", "%d/%m/%Y"):
        try:
            return datetime.strptime(due_clean, fmt)
        except ValueError:
            continue
            
    months = {
        "january": 1, "jan": 1, "february": 2, "feb": 2, "march": 3, "mar": 3,
        "april": 4, "apr": 4, "may": 5, "june": 6, "jun": 6, "july": 7, "jul": 7,
        "august": 8, "aug": 8, "september": 9, "sept": 9, "sep": 9, "october": 10, "oct": 10,
        "november": 11, "nov": 11, "december": 12, "dec": 12
    }
    
    cleaned_words = []
    for word in due_clean.replace(",", " ").replace(".", " ").split():
        if any(c.isdigit() for c in word):
            num_str = "".join(c for c in word if c.isdigit())
            cleaned_words.append(num_str)
        else:
            cleaned_words.append(word)
            
    month_val = None
    day_val = None
    
    for word in cleaned_words:
        if word in months:
            month_val = months[word]
        elif word.isdigit():
            val = int(word)
            if 1 <= val <= 31:
                day_val = val
                
    if month_val is not None and day_val is not None:
        try:
            return datetime(2026, month_val, day_val)
        except ValueError:
            pass
            
    return datetime(current_year + 5, 12, 31)

def build_schedule() -> str:
    """
    Looks at all tasks stored in memory, sorts them by deadline, and schedules
    study periods leading up to each task's deadline.
    """
    stored_tasks = get_tasks_from_memory()
    if not stored_tasks:
        return "Warning: No tasks found in memory to schedule. Add tasks first."
        
    # Sort tasks by their parsed due dates
    sorted_tasks = sorted(stored_tasks, key=lambda t: parse_due_date(t["due"]))
    
    # Anchor date for schedule start
    start_date = datetime(2026, 8, 27)
    
    schedule_lines = ["=== Study Planner Schedule ==="]
    
    for i, task in enumerate(sorted_tasks, 1):
        due_date = parse_due_date(task["due"])
        
        if due_date <= start_date:
            study_start = start_date
            study_end = start_date
        else: 
            study_start = start_date
            study_end = due_date
            
        schedule_lines.append(
            f"{i}. Task: {task['name']}\n"
            f"   Due Date: {task['due']}\n"
            f"   Study Period: {study_start.strftime('%B %d, %Y')} to {study_end.strftime('%B %d, %Y')}"
        )
        
        start_date = due_date + timedelta(days=1)
        
    return "\n\n".join(schedule_lines)

---  
## Cell 7 — Agent Setup

Here we define the `StudyPlannerAgent` class. It loads conversational history and specifies Gemini tools.

In [ ]:
class StudyPlannerAgent:
    def __init__(self, model_name: str = "gemini-2.5-flash"):
        """
        Initialize the study planner agent.
        """
        api_key = os.getenv("GEMINI_API_KEY")
        if not api_key:
            raise ValueError(
                "GEMINI_API_KEY is missing from environment. "
                "Please configure a '.env' file in the project folder with: "
                "GEMINI_API_KEY=your_actual_api_key"
            )
            
        # Instantiate the Gemini client
        self.client = genai.Client(api_key=api_key)
        self.model_name = model_name
        self.history = []
        
        # Clear task memory at start of agent session
        clear_tasks_in_memory()
        self.tools = [add_task, build_schedule]

---  
## Cell 8 — Agent Execution / Plan-Act Logic

We implement the core execution loop containing the plan-act logic. The agent sends conversation history to Gemini, parses tool execution decisions, executes them, logs the trace, and feeds the results back to the model.

In [ ]:
def run_agent_turn(self, user_input: str) -> str:
    """
    Execute a single user turn in the plan-act loop.
    """
    print(f"\n==========================================")
    print(f"USER: '{user_input}'")
    print(f"==========================================")
    
    # 1. Append user input as types.Content to conversation history
    user_content = types.Content(
        role="user",
        parts=[types.Part.from_text(text=user_input)]
    )
    self.history.append(user_content)
    
    step_count = 1
    
    # 2. Plan-Act Loop
    while True:
        print(f"\n[Step {step_count}: Plan] Sending conversation history to model...")
        
        response = self.client.models.generate_content(
            model=self.model_name,
            contents=self.history,
            config=types.GenerateContentConfig(
                tools=self.tools,
                automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True),
                system_instruction=(
                    "You are a study planner AI agent.\n"
                    "Your job is to help the user schedule study blocks around exams and assignments.\n"
                    "You have two tools:\n"
                    "- add_task(name, due): Saves a task and deadline to memory.\n"
                    "- build_schedule(): Sorts all saved tasks and generates a study schedule.\n\n"
                    "Guidelines:\n"
                    "1. When the user specifies new tasks, call add_task for each one.\n"
                    "2. When they ask you to plan, structure, or schedule, call build_schedule.\n"
                    "3. Do not formulate a final reply until you have called the necessary tools and received results.\n"
                    "4. Always present the final schedule clearly if it is returned by the tool."
                )
            )
        )
        
        # 3. Action Phase
        if response.function_calls:
            print(f"[Step {step_count}: Act] Agent decided to call tools:")
            
            # Save model's tool calls to history
            model_function_calls = types.Content(
                role="model",
                parts=[types.Part.from_function_call(name=call.name, args=call.args) for call in response.function_calls]
            )
            self.history.append(model_function_calls)
            
            # Execute function calls locally
            for call in response.function_calls:
                print(f"  -> Tool Called: {call.name}")
                print(f"  -> Tool Inputs: {call.args}")
                
                if call.name == "add_task":
                    name = call.args.get("name")
                    due = call.args.get("due")
                    tool_result = add_task(name=name, due=due)
                elif call.name == "build_schedule":
                    tool_result = build_schedule()
                else:
                    tool_result = f"Error: Tool '{call.name}' is not recognized."
                    
                print(f"  -> Tool Result: {tool_result}")
                
                # Save tool output to history
                tool_response_content = types.Content(
                    role="tool",
                    parts=[types.Part.from_function_response(
                        name=call.name,
                        response={"result": tool_result}
                    )]
                )
                self.history.append(tool_response_content)
            
            step_count += 1
            
        else:
            # 4. Final Answer Phase
            final_answer = response.text
            print(f"[Step {step_count}: Final Response] {final_answer}")
            
            # Save final response to history
            model_final_content = types.Content(
                role="model",
                parts=[types.Part.from_text(text=final_answer)]
            )
            self.history.append(model_final_content)
            
            return final_answer

# Attach the run function to our StudyPlannerAgent class dynamically
StudyPlannerAgent.run = run_agent_turn
print("Agent execution loop set up!")

---  
## Cell 9 — Demonstration initialization

We now instantiate the agent. Make sure that you have placed a `.env` file containing your `GEMINI_API_KEY` in the workspace directory.

In [ ]:
try:
    agent = StudyPlannerAgent()
    print("Study Planner Agent successfully initialized and ready to run!")
except Exception as e:
    print(f"Initialization Error: {e}")
    print("Ensure GEMINI_API_KEY is correctly set in your .env file.")

---  
## Example 1: Planning Multiple Deadlines

We instruct the agent to plan our study schedule because we have a **Java exam on September 5** and a **DBMS assignment due September 2**.

Watch the agent trace:
1. It decides to call `add_task` for both deadlines.
2. It runs them and returns the results.
3. It then decides to call `build_schedule` to compute the calendar.
4. It displays the schedule in the final response.

In [ ]:
# Run Example 1
try:
    agent.run("I have a Java exam on September 5 and a DBMS assignment due September 2. Plan my study schedule.")
except Exception as e:
    print(f"Execution Error: {e}")

---  
## Example 2: Conversation Session Memory

We continue the conversation by asking a follow-up query: *"What should I study first and when does it start?"*

We do not repeat any dates or subject names. The agent reads the conversation history from its memory, understands what tasks were added previously, and correctly answers our question.

In [ ]:
# Run Example 2
try:
    agent.run("What should I study first and when does it start?")
except Exception as e:
    print(f"Execution Error: {e}")

---  
## Example 3: Re-Planning with an Urgent Task

We add a new urgent task: a **Python quiz due August 30**. The agent should:
1. Save the Python quiz to memory.
2. Re-run `build_schedule`.
3. Create an updated schedule incorporating all three tasks sorted chronologically: Python (Aug 30), DBMS (Sept 2), and Java (Sept 5).

In [ ]:
# Run Example 3
try:
    agent.run("I also have a Python quiz due August 30. Re-plan my schedule.")
except Exception as e:
    print(f"Execution Error: {e}")

---  
## Final Cell — Requirement Verification

This project satisfies all course requirements:
- **✓ One AI Agent**: Defined in `StudyPlannerAgent`.
- **✓ Two Tools**: Local functions `add_task` and `build_schedule`.
- **✓ Multi-step plan-act behavior**: Handled in a manual loop that continues until all required tool executions complete.
- **✓ Memory across turns**: The conversational context list remembers previous turns, and the in-memory task database retains tasks.
- **✓ 2–3 demonstration examples**: Demonstrated above (Initial plan, Context check, Re-planning).
- **✓ Tool calls visible in notebook**: The step-by-step logs print out details, arguments, and outcomes for every function execution.